# pytest와 ruff 실습

이 노트북은 두 도구가 실제로 어떻게 동작하는지 눈으로 확인하도록 만들었습니다.

- **pytest**: 코드가 기대대로 동작하는지 자동으로 검사하는 테스트 도구입니다.
- **ruff**: 잠재적 오류와 스타일 문제를 찾고(`check`), 코드 모양을 정리하는(`format`) 도구입니다.

셀을 위에서부터 **순서대로** 실행하세요. Part 1~2의 실습은 임시 폴더(샘플 프로젝트)에서 실행되므로 실제 프로젝트 파일은 바뀌지 않습니다. Part 3만 실제 `p3_2Expense` 프로젝트를 읽습니다.

노트북 커널은 **py311**이어야 합니다.

## 0. 준비: 도구 확인

In [1]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

PY = sys.executable  # 이 노트북 커널의 파이썬 (py311이어야 합니다)
print("파이썬:", PY)


def run(args, cwd=None):
    # 명령을 실행하고 출력과 종료 코드를 보여준다.
    # 종료 코드 0은 성공, 0이 아니면 실패하거나 문제를 발견했다는 뜻이다.
    result = subprocess.run(
        [PY, "-m", *args], cwd=cwd, capture_output=True, text=True, encoding="utf-8"
    )
    print(result.stdout + result.stderr)
    print(f"[종료 코드: {result.returncode}]")
    return result.returncode


run(["pytest", "--version"])
run(["ruff", "--version"])

파이썬: c:\Users\great\anaconda3\envs\py311\python.exe
pytest 9.2.0

[종료 코드: 0]
ruff 0.17.3

[종료 코드: 0]


0

## 1. pytest 기초

pytest는 아래 규칙만 지키면 테스트를 자동으로 찾아 실행합니다.

- 파일 이름이 `test_`로 시작하거나 `_test.py`로 끝남
- 테스트 함수 이름이 `test_`로 시작함
- 검사는 `assert` 문으로 씁니다. 조건이 거짓이면 그 테스트는 실패합니다.

### 1-1. 샘플 폴더와 파일 만들기

In [2]:
SANDBOX = Path(tempfile.mkdtemp(prefix="practice_"))
print("샘플 폴더:", SANDBOX)


def write(name, code):
    # 샘플 파일을 만든다. dedent로 여는 줄의 들여쓰기를 정리한다.
    path = SANDBOX / name
    path.write_text(textwrap.dedent(code), encoding="utf-8")
    return path


write("calc.py", '''
    def add(a, b):
        return a + b

    def divide(a, b):
        if b == 0:
            raise ZeroDivisionError("0으로 나눌 수 없습니다")
        return a / b
''')
print(SANDBOX.joinpath("calc.py").read_text(encoding="utf-8"))

샘플 폴더: C:\Users\great\AppData\Local\Temp\practice_1ymyqe05

def add(a, b):
    return a + b

def divide(a, b):
    if b == 0:
        raise ZeroDivisionError("0으로 나눌 수 없습니다")
    return a / b



### 1-2. 통과하는 테스트와 예외를 검사하는 테스트

`pytest.raises`는 "이 코드가 특정 예외를 일으켜야 한다"는 뜻입니다. 예외가 나지 않으면 테스트가 실패합니다.

In [3]:
write("test_calc.py", '''
    import pytest
    from calc import add, divide

    def test_add():
        assert add(2, 3) == 5

    def test_divide_by_zero_raises():
        with pytest.raises(ZeroDivisionError):
            divide(1, 0)
''')

run(["pytest", "test_calc.py", "-v"], cwd=SANDBOX)

============================= test session starts =============================
platform win32 -- Python 3.11.13, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\great\anaconda3\envs\py311\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\great\AppData\Local\Temp\practice_1ymyqe05
plugins: anyio-4.10.0, dash-3.3.0
collecting ... collected 2 items

test_calc.py::test_add PASSED                                            [ 50%]
test_calc.py::test_divide_by_zero_raises PASSED                          [100%]

============================== warnings summary ===============================
..\..\..\..\anaconda3\envs\py311\Lib\site-packages\dash\testing\plugin.py:104
  c:\Users\great\anaconda3\envs\py311\Lib\site-packages\dash\testing\plugin.py:104: PytestRemovedIn10Warning: The hookimpl pytest_addhooks uses old-style configuration options (marks or attributes).
  Please use the pytest.hookimpl(tryfirst=True) decorator instead
   to configure the hooks.
   See https://docs.pytest.org/en/latest/depr

0

결과에서 `PASSED`는 통과, `FAILED`는 실패입니다. `-v`를 빼면 `.`(통과)과 `F`(실패)로 간단히 표시됩니다.

### 1-3. 실패 메시지 읽기

일부러 틀린 기대값을 넣어 봅니다.

In [4]:
write("test_fail.py", '''
    from calc import add

    def test_add_wrong_expectation():
        assert add(2, 3) == 6
''')

run(["pytest", "test_fail.py"], cwd=SANDBOX)

============================= test session starts =============================
platform win32 -- Python 3.11.13, pytest-9.2.0, pluggy-1.7.0
rootdir: C:\Users\great\AppData\Local\Temp\practice_1ymyqe05
plugins: anyio-4.10.0, dash-3.3.0
collected 1 item

test_fail.py F                                                           [100%]

================================== FAILURES ===================================
_________________________ test_add_wrong_expectation __________________________

    def test_add_wrong_expectation():
>       assert add(2, 3) == 6
E       assert 5 == 6
E        +  where 5 = add(2, 3)

test_fail.py:5: AssertionError
============================== warnings summary ===============================
..\..\..\..\anaconda3\envs\py311\Lib\site-packages\dash\testing\plugin.py:104
  c:\Users\great\anaconda3\envs\py311\Lib\site-packages\dash\testing\plugin.py:104: PytestRemovedIn10Warning: The hookimpl pytest_addhooks uses old-style configuration options (marks or attrib

1

실패 메시지에 `assert add(2, 3) == 6`과 실제 계산값 `5`가 함께 나옵니다. pytest가 `assert` 문을 분석해서 중간 값을 보여 주기 때문에, 별도의 `print`를 넣지 않아도 원인을 알 수 있습니다.

### 1-4. fixture와 parametrize

- **fixture**: 테스트에 필요한 준비 데이터를 만드는 함수입니다. 테스트 함수의 인자 이름과 같은 fixture를 pytest가 자동으로 넣어 줍니다. 테스트마다 새로 만들어지므로 테스트끼리 영향을 주지 않습니다.
- **parametrize**: 같은 테스트를 여러 입력값으로 반복 실행합니다.

In [5]:
write("test_advanced.py", '''
    import pytest
    from calc import add

    @pytest.fixture
    def numbers():
        # 테스트마다 새로 만들어지는 데이터
        return [1, 2, 3]

    def test_sum_uses_fixture(numbers):
        assert sum(numbers) == 6

    @pytest.mark.parametrize("a, b, expected", [
        (1, 1, 2),
        (0, 5, 5),
        (-1, 1, 0),
    ])
    def test_add_many(a, b, expected):
        assert add(a, b) == expected
''')

run(["pytest", "test_advanced.py", "-v"], cwd=SANDBOX)

============================= test session starts =============================
platform win32 -- Python 3.11.13, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\great\anaconda3\envs\py311\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\great\AppData\Local\Temp\practice_1ymyqe05
plugins: anyio-4.10.0, dash-3.3.0
collecting ... collected 4 items

test_advanced.py::test_sum_uses_fixture PASSED                           [ 25%]
test_advanced.py::test_add_many[1-1-2] PASSED                            [ 50%]
test_advanced.py::test_add_many[0-5-5] PASSED                            [ 75%]
test_advanced.py::test_add_many[-1-1-0] PASSED                           [100%]

============================== warnings summary ===============================
..\..\..\..\anaconda3\envs\py311\Lib\site-packages\dash\testing\plugin.py:104
  c:\Users\great\anaconda3\envs\py311\Lib\site-packages\dash\testing\plugin.py:104: PytestRemovedIn10Warning: The hookimpl pytest_addhooks uses old-style configuration options (

0

`-v` 출력에서 `test_add_many`가 입력값 세 개에 대해 따로따로 실행된 것을 확인하세요. 하나가 실패해도 나머지 입력값의 결과는 그대로 나옵니다.

## 2. ruff 기초

ruff는 두 가지 명령을 주로 씁니다.

- `ruff check`: 규칙 위반을 찾습니다. 잠재적 버그, 쓰지 않는 코드, 스타일 문제가 대상입니다. `--fix`를 붙이면 자동으로 고칠 수 있는 것은 고칩니다.
- `ruff format`: 들여쓰기, 따옴표, 줄바꿈 같은 모양만 정리합니다. 코드의 의미는 바꾸지 않습니다.

규칙은 `select`로 고릅니다. 예를 들어 `E`는 스타일, `F`는 pyflakes 규칙(사용하지 않는 import와 변수 등), `B`는 흔한 실수, `I`는 import 정렬입니다. 이 프로젝트의 `ruff.toml`은 `E`, `W`, `F`, `I`, `B`, `UP`을 켜 두었습니다. 샘플 폴더에는 설정 파일이 없으므로 기본 규칙(`E4`, `E7`, `E9`, `F`)으로 검사합니다.

### 2-1. 문제가 있는 코드 찾기

In [6]:
write("messy.py", '''
    import os
    import json

    def greet(name):
        unused = 2
        if name == None:
            return "hi"+name
        return f"hello {name}"
''')

run(["ruff", "check", "messy.py", "--output-format", "concise"], cwd=SANDBOX)

messy.py:2:1: I001 [*] Import block is un-sorted or un-formatted
messy.py:2:8: F401 [*] `os` imported but unused
messy.py:3:8: F401 [*] `json` imported but unused
messy.py:6:5: F841 Local variable `unused` is assigned to but never used
Found 4 errors.
[*] 3 fixable with `--fix` (1 unsafe fix available with `--fix --unsafe-fixes`).

[종료 코드: 1]


1

위반이 하나라도 있으면 종료 코드가 1이 됩니다. 각 줄 앞의 `F401`, `F841`, `E711`은 **규칙 번호**입니다. 나중에 `ruff.toml`에서 이 번호로 규칙을 켜거나 끌 수 있습니다. 메시지를 보고 규칙 번호를 검색하면 왜 문제인지 설명하는 페이지를 찾을 수 있습니다.

### 2-2. 자동 수정 (`--fix`)

In [7]:
run(["ruff", "check", "messy.py", "--fix", "--output-format", "concise"], cwd=SANDBOX)
print("--- 수정 후 messy.py ---")
print(SANDBOX.joinpath("messy.py").read_text(encoding="utf-8"))

messy.py:4:5: F841 Local variable `unused` is assigned to but never used
Found 3 errors (2 fixed, 1 remaining).
1 unsafe fix available with `--fix --unsafe-fixes`.

[종료 코드: 1]
--- 수정 후 messy.py ---


def greet(name):
    unused = 2
    if name == None:
        return "hi"+name
    return f"hello {name}"



자동 수정이 안 되는 항목은 그대로 남습니다. `--fix`를 실행한 뒤에도 남은 경고는 직접 고쳐야 합니다.

### 2-3. 모양 정리 (`format`)

`--diff`는 파일을 실제로 바꾸지 않고 바뀔 내용만 보여 줍니다. 바꿀 것이 있으면 종료 코드가 1이 됩니다.

In [8]:
write("style.py", '''
    def greet( name ):
        return {'msg':'hello '+name}
''')

run(["ruff", "format", "--diff", "style.py"], cwd=SANDBOX)

--- style.py
+++ style.py
@@ -1,3 +1,2 @@
-
-def greet( name ):
-    return {'msg':'hello '+name}
+def greet(name):
+    return {"msg": "hello " + name}

1 file would be reformatted

[종료 코드: 1]


1

### 2-4. 특정 줄에서 규칙 무시하기 (`noqa`)

줄 끝에 `# noqa: 규칙번호`를 쓰면 그 줄의 해당 경고만 무시합니다. 꼭 필요한 이유가 있을 때만 쓰세요. 남용하면 규칙을 켜 둔 의미가 없어집니다.

In [9]:
write("ignore.py", '''
    import os  # noqa: F401
''')

run(["ruff", "check", "ignore.py", "--output-format", "concise"], cwd=SANDBOX)

All checks passed!

[종료 코드: 0]


0

## 3. 실제 프로젝트에 적용하기

이제 실제 `p3_2Expense` 폴더에서 같은 도구를 씁니다. 이 노트북을 `p3_2Expense` 폴더에서 열었다면 `Path.cwd()`가 그 폴더를 가리킵니다.

- pytest는 `test_expense_manager.py`를 찾아 실행합니다. 테스트는 임시 폴더를 쓰므로 실제 `data.json`은 건드리지 않습니다.
- ruff는 `ruff.toml` 설정(줄 길이 100, 규칙 `E`, `W`, `F`, `I`, `B`, `UP`)을 따릅니다.

캐시 폴더가 생기지 않도록 `-p no:cacheprovider`와 `--no-cache`를 붙였습니다.

In [10]:
PROJECT = Path.cwd()
print("프로젝트 폴더:", PROJECT)

run(["pytest", "-q", "-p", "no:cacheprovider"], cwd=PROJECT)
run(["ruff", "check", ".", "--no-cache", "--output-format", "concise"], cwd=PROJECT)

프로젝트 폴더: c:\Users\great\VibeProj\DG_AI\p3_2Expense
.................                                                        [100%]
============================== warnings summary ===============================
..\..\..\anaconda3\envs\py311\Lib\site-packages\dash\testing\plugin.py:104
  c:\Users\great\anaconda3\envs\py311\Lib\site-packages\dash\testing\plugin.py:104: PytestRemovedIn10Warning: The hookimpl pytest_addhooks uses old-style configuration options (marks or attributes).
  Please use the pytest.hookimpl(tryfirst=True) decorator instead
   to configure the hooks.
   See https://docs.pytest.org/en/latest/deprecations.html#configuring-hook-specs-impls-using-markers
    @pytest.mark.tryfirst

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
17 passed, 1 warning in 0.04s

[종료 코드: 0]
All checks passed!

[종료 코드: 0]


0

## 4. 중급 포인트

- **테스트 격리**: 테스트는 서로 독립적이어야 합니다. 실제 파일 대신 `tmp_path` 같은 임시 경로 fixture를 쓰면 매번 깨끗한 상태에서 시작합니다. 이 프로젝트의 `test_expense_manager.py`가 이 방식입니다.
- **fixture 범위(scope)**: `@pytest.fixture(scope="module")`처럼 쓰면 파일 하나당 한 번만 만들어 재사용합니다. 기본값인 `function`은 테스트마다 새로 만듭니다. 느린 준비 작업에만 범위를 넓히세요.
- **conftest.py**: 여러 테스트 파일이 같이 쓰는 fixture를 넣는 파일입니다. 따로 import하지 않아도 pytest가 자동으로 찾습니다.
- **marker**: `@pytest.mark.slow`처럼 표시를 달아 두고 `pytest -m "not slow"`로 골라 실행할 수 있습니다. 표시를 쓰려면 설정 파일에 등록하는 것이 좋습니다.
- **ruff 규칙 점진 적용**: 처음부터 모든 규칙을 켜면 경고가 너무 많습니다. `E`와 `F`처럼 작게 시작해서 늘려 가세요. 특정 파일만 예외로 두려면 `[lint.per-file-ignores]`를 씁니다.
- **실행 순서**: 보통 `ruff format` → `ruff check --fix` → `pytest` 순서가 편합니다. 이 세 가지를 한 번에 돌리는 스크립트나 git pre-commit 훅으로 묶으면 실수를 줄일 수 있습니다.

## 5. 연습문제

1. `calc.py`에 `divide(6, 3) == 2`를 검사하는 테스트를 `test_calc.py`에 추가하고 `pytest`로 확인해 보세요.
2. 일부러 틀린 기대값을 가진 테스트를 만들고, 실패 메시지에서 기대값과 실제 값이 어디에 나오는지 찾아보세요.
3. `messy.py` 같은 파일을 새로 만들어 `ruff check`로 찾고, `--fix` 뒤에 무엇이 남는지 비교해 보세요.
4. `parametrize`에 입력값을 한 줄 더 추가해 보고, `-v` 출력이 어떻게 바뀌는지 확인하세요.

아래 셀에 코드를 작성해 실행하세요. `run([...], cwd=SANDBOX)`로 명령을 실행할 수 있고, `write(이름, 내용)`으로 샘플 파일을 만들 수 있습니다.

In [ ]:
# 연습 1 작성 공간
# 예시: write("test_divide.py", '''...''')로 테스트 파일을 만든 뒤
# run(["pytest", "test_divide.py", "-v"], cwd=SANDBOX)로 실행합니다.